# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

In [1]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Fetch HF_TOKEN from the Colab environment/secrets.
HF_TOKEN = userdata.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named 'HF_TOKEN'."
    )

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Authenticate DuckDB with Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# ML-04 windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("HF_TOKEN fetched successfully.")
print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")

HF_TOKEN fetched successfully.
Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026


## 1. Build the feature vector

The feature vector represents what was knowable at the decision point (the end of February 2026).

I use five features covering search demand, recent performance, visibility, and content freshness:

- `gsc_impressions_90d`: measured search impressions over the available 90-day window.
- `gsc_clicks_90d`: measured search clicks over the available 90-day window.
- `avg_position_90d`: impression-weighted average search position.
- `content_age_days`: age of the content at the decision point.
- `days_since_last_update`: time since the content was last updated.

These are used for decision-support and are not treated as causal drivers of future performance.

In [5]:
# Build the February 2026 feature frame.
# All fields below are available at the decision point.

feb_features = con.sql(f"""
WITH feb_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions_90d,
        SUM(gsc_clicks) AS gsc_clicks_90d,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_90d

    FROM {FEB}
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    f.gsc_impressions_90d,
    f.gsc_clicks_90d,
    f.avg_position_90d,

    datediff('day', c.content_created_date, DATE '2026-02-28') AS content_age_days,
    datediff('day', c.content_updated_date, DATE '2026-02-28') AS days_since_last_update

FROM feb_agg f

JOIN read_parquet('{DIM_CONTENT}') c
    ON f.content_hash_id = c.content_hash_id

WHERE
    f.gsc_impressions_90d >= 100
    AND f.gsc_clicks_90d >= 3
    AND c.is_published IS TRUE
    AND c.content_created_date <= DATE '2026-02-28'
""").df()

print("Feature frame shape:", feb_features.shape)
display(feb_features.head())

Feature frame shape: (29700, 7)


,client_hash_id,content_hash_id,gsc_impressions_90d,gsc_clicks_90d,avg_position_90d,content_age_days,days_since_last_update
0,client_3ffa76342f366962,content_5573434837db89c5,198.0,6.0,7.318182,153,-81
1,client_3ffa76342f366962,content_7b17975c58745266,102.0,5.0,4.450980,29,-81
2,client_3ffa76342f366962,content_b89167cd03d6ffc1,178.0,6.0,3.393258,29,-81
3,client_e547b89c05043229,content_eb42160708b4da7c,4802.0,5.0,9.917118,344,-104
4,client_e547b89c05043229,content_4a630add28e014a5,7842.0,8.0,19.141801,344,-104


In [6]:
feature_cols = [
    "gsc_impressions_90d",
    "gsc_clicks_90d",
    "avg_position_90d",
    "content_age_days",
    "days_since_last_update",
]

print("Feature columns:")
for col in feature_cols:
    print(f"- {col}: {feb_features[col].dtype}")

print("\nMissing values:")
display(
    feb_features[feature_cols]
    .isna()
    .sum()
    .to_frame("missing_count")
)

print("\nFeature summary:")
display(feb_features[feature_cols].describe().T)

Feature columns:
- gsc_impressions_90d: float64
- gsc_clicks_90d: float64
- avg_position_90d: float64
- content_age_days: int64
- days_since_last_update: int64

Missing values:


,missing_count
gsc_impressions_90d,0
gsc_clicks_90d,0
avg_position_90d,0
content_age_days,0
days_since_last_update,0



Feature summary:


,count,mean,std,min,25%,50%,75%,max
gsc_impressions_90d,29700.0,4878.206195,7771.160910,100.00000,1322.750000,2561.000000,5370.000000,167303.000000
gsc_clicks_90d,29700.0,18.595084,46.995007,3.00000,4.000000,8.000000,17.000000,3310.000000
avg_position_90d,29700.0,7.771974,7.020005,0.03963,3.444262,5.422985,8.740648,69.695192
content_age_days,29700.0,182.278586,109.263573,3.00000,103.000000,184.000000,235.000000,463.000000
days_since_last_update,29700.0,-90.100034,44.892687,-128.00000,-124.000000,-107.000000,-81.000000,233.000000


In [7]:
# Numeric missing-value handling.
# Median values are calculated only from the February feature window.

X = feb_features[feature_cols].copy()

for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].fillna(X[col].median())

print("Final feature matrix shape:", X.shape)
print("Remaining missing values:", int(X.isna().sum().sum()))

display(X.head())

Final feature matrix shape: (29700, 5)
Remaining missing values: 0


,gsc_impressions_90d,gsc_clicks_90d,avg_position_90d,content_age_days,days_since_last_update
0,198.0,6.0,7.318182,153,-81
1,102.0,5.0,4.450980,29,-81
2,178.0,6.0,3.393258,29,-81
3,4802.0,5.0,9.917118,344,-104
4,7842.0,8.0,19.141801,344,-104


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `gsc_impressions_90d` | Search impressions measured over the available 90-day window | Median of February feature window | Before the decision point |
| `gsc_clicks_90d` | Search clicks measured over the available 90-day window | Median of February feature window | Before the decision point |
| `avg_position_90d` | Impression-weighted average search position | Median of February feature window | Before the decision point |
| `content_age_days` | Age of the content at the decision point | Median of February feature window | Before the decision point |
| `days_since_last_update` | Days since the content was last updated | Median of February feature window | Before the decision point |

No March outcome information is used to construct these features.

## 3. The leakage hunt

I tested the feature vector for three leakage risks:

1. **Future-window leakage:** using March performance when the decision is made at the end of February.
2. **Label-derived leakage:** using a field that directly reveals the future outcome.
3. **Product-decision leakage:** using an existing health, priority, action, or refresh field as an input.

For the deliberate test, I added March clicks as a feature. Because the label is defined from March clicks, this produced an apparent accuracy of 1.000. This result is intentionally invalid and demonstrates why future outcome data cannot be part of the feature vector.

I removed the leaked field and retained only the five February-side features.

In [8]:
label_derived = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "went_dark",
}

future_fields = {
    "march_clicks",
    "march_impressions",
    "future_decline",
    "future_clicks",
    "future_impressions",
}

product_decision = {
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
}

forbidden = label_derived | future_fields | product_decision

leakage_matches = sorted(set(X.columns) & forbidden)

print("Feature columns:", list(X.columns))
print("Forbidden-column matches:", leakage_matches)

assert not leakage_matches, (
    f"Potential leakage found: {leakage_matches}"
)

print("Leakage column check: PASS")

Feature columns: ['gsc_impressions_90d', 'gsc_clicks_90d', 'avg_position_90d', 'content_age_days', 'days_since_last_update']
Forbidden-column matches: []
Leakage column check: PASS


### Deliberate leakage experiment

In [14]:
# Deliberate leakage experiment.
# We intentionally use March outcome information as a feature.
# This information is NOT available at the February decision point.

# Build the March outcome for the same page/client pairs.
march_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# Join the future outcome onto the February feature frame.
leak_test = feb_features.merge(
    march_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Keep only rows where March GSC was actually measured.
leak_test = leak_test[
    leak_test["clicks_mar"].notna()
].copy()

# This is the actual future label.
leak_test["went_dark"] = (
    leak_test["clicks_mar"] == 0
).astype(int)

# DELIBERATE LEAK:
# We give the model the future March click count.
# It directly reveals the label in this experiment.
leak_test["future_clicks_leak"] = leak_test["clicks_mar"]

# A deliberately leaked prediction.
leak_test["leaked_prediction"] = (
    leak_test["future_clicks_leak"] == 0
).astype(int)

leak_accuracy = (
    leak_test["leaked_prediction"]
    == leak_test["went_dark"]
).mean()

print("Deliberate leakage test")
print("Leaked feature: future_clicks_leak")
print("Rows tested:", len(leak_test))
print(f"Apparent accuracy: {leak_accuracy:.3f}")

assert leak_accuracy == 1.0

print("\nThe perfect score is invalid because March is in the future.")

Deliberate leakage test
Leaked feature: future_clicks_leak
Rows tested: 29353
Apparent accuracy: 1.000

The perfect score is invalid because March is in the future.


### Remove the leakage

In [15]:
# Remove all future/label-derived columns.
X_clean = feb_features[feature_cols].copy()

# Handle missing numeric values using only the feature window.
for col in feature_cols:
    X_clean[col] = pd.to_numeric(X_clean[col], errors="coerce")
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

print("Final clean feature vector:")
display(X_clean.head())

print("\nFinal feature columns:")
print(list(X_clean.columns))

# Confirm no future or label-derived fields remain.
for forbidden_col in [
    "clicks_mar",
    "future_clicks_leak",
    "went_dark",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
]:
    assert forbidden_col not in X_clean.columns, (
        f"Leakage/product field still present: {forbidden_col}"
    )

assert X_clean.shape[1] == 5
assert X_clean.isna().sum().sum() == 0

print("\nClean feature-vector check: PASS")

Final clean feature vector:


,gsc_impressions_90d,gsc_clicks_90d,avg_position_90d,content_age_days,days_since_last_update
0,198.0,6.0,7.318182,153,-81
1,102.0,5.0,4.450980,29,-81
2,178.0,6.0,3.393258,29,-81
3,4802.0,5.0,9.917118,344,-104
4,7842.0,8.0,19.141801,344,-104



Final feature columns:
['gsc_impressions_90d', 'gsc_clicks_90d', 'avg_position_90d', 'content_age_days', 'days_since_last_update']

Clean feature-vector check: PASS


### 4. What I excluded and why

- `trend_direction` — label-derived information; using it would directly leak the observed outcome.
- `trend_pct` — label-derived trend information; not available as an honest predictive feature.
- March 2026 performance fields — future information relative to the February decision point.
- `is_declining_label` — directly derived from `trend_direction`.
- `health_score` — existing product decision field rather than an independent input signal.
- `priority_score` — already represents a prioritization decision and could leak downstream logic.
- `action_type` — product decision output, not an independent feature.
- `refresh_tier` — existing decision category that should not be used as an input.
- `client_hash_id` — grouping/join/split context only, not a predictive feature.
- `content_hash_id` — identifier only; using it as a feature would not represent a meaningful content signal.
- `March outcome data` — belongs to the future outcome window and must remain unavailable at prediction time.

The final vector therefore contains only five features that are measurable before the decision point.

In [16]:
# ML-05 final self-check

assert X_clean.shape[1] == 5

assert set(X_clean.columns) == {
    "gsc_impressions_90d",
    "gsc_clicks_90d",
    "avg_position_90d",
    "content_age_days",
    "days_since_last_update",
}

assert X_clean.isna().sum().sum() == 0

for forbidden_col in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "went_dark",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
]:
    assert forbidden_col not in X_clean.columns

print("ML-05 self-check: PASS")
print("Feature count:", X_clean.shape[1])
print("Rows:", X_clean.shape[0])
print("Missing values:", int(X_clean.isna().sum().sum()))
print("Leakage/product-decision fields present: 0")

ML-05 self-check: PASS
Feature count: 5
Rows: 29700
Missing values: 0
Leakage/product-decision fields present: 0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.